# Aggregate a time series

Turn a long series into a few typical periods. Every other guide here is a variation on this
call.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}
data.head()

## Aggregate

Two parameters carry the decision:

* **`n_clusters`** — how many typical periods to keep.
* **`period_duration`** — the length of one: `"1D"`, `"1W"`, or a number of hours.

Add **`temporal_resolution`** (e.g. `"15min"`) if your index is irregular or its step cannot be
inferred.

In [ ]:
result = tsam.aggregate(data, n_clusters=8, period_duration="1D")
print("reduced", len(data), "hours to", result.n_clusters, "typical days")

## Read the outputs

Three pieces are what a downstream model consumes:

* **`cluster_representatives`** — the typical period profiles.
* **`cluster_counts`** — how many real periods each stands for (its weight).
* **`accuracy`** — RMSE / MAE per column.

In [ ]:
print("counts:", result.cluster_counts)
print("\nper-column RMSE:")
print(result.accuracy.rmse.round(3).to_string())
result.cluster_representatives.head()

In [ ]:
result.plot.compare(
    columns=["Load"],
    time_slice=slice("2010-01-11", "2010-01-17"),
    color="source",
    units=UNITS,
    title="One week: original vs. reconstructed Load",
)

## From here

| To… | Go to |
|---|---|
| make it smaller *within* each period | [Segmentation](segmentation.ipynb) |
| hit a target size | [How small can you go?](tuning.ipynb) |
| change how periods are grouped | [Clustering methods](clustering_methods.ipynb) |
| change what each typical period keeps | [Representations](representations.ipynb) |
| keep the peak day exactly | [Extreme periods](extreme_periods.ipynb) |
| know what it will cost to run | [How long will this take?](runtime.ipynb) |
| hand the result to a model | [Optimization workflow](optimization_workflow.ipynb) |

Unsure which lever you need? [Choosing a method](../tutorials/choosing_a_method.ipynb) walks all
four on one dataset.